# Vision Model Test — BMS Screenshot Extraction

Test free vision model APIs against BMS screenshots.

**Supported providers:**
- `gemini` — Google Gemini Flash (free tier, best quality)
- `groq` — Groq LLaMA Vision (free tier, fastest)
- `openrouter` — OpenRouter (free models available)
- `together` — Together AI ($25 free credits)
- `custom` — any OpenAI-compatible endpoint (uses LLM_* from .env)

**Before running:** make sure your API key is set in `.env` or in the config cell below.

In [ ]:
import base64
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root
PROJECT_ROOT = Path(".").resolve().parent
load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")

## 1. Choose Provider and Model

In [ ]:
# ── Pick one provider ──────────────────────────────────────────────────────
PROVIDER = "gemini"   # gemini | groq | openrouter | together | custom

PROVIDER_CONFIGS = {
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai",
        "model": "gemini-2.0-flash",
        "api_key_env": "GEMINI_API_KEY",
    },
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "model": "llama-4-scout-17b-16e-instruct",
        "api_key_env": "GROQ_API_KEY",
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "model": "qwen/qwen2-vl-7b-instruct:free",
        "api_key_env": "OPENROUTER_API_KEY",
    },
    "together": {
        "base_url": "https://api.together.xyz/v1",
        "model": "Qwen/Qwen2-VL-7B-Instruct",
        "api_key_env": "TOGETHER_API_KEY",
    },
    "custom": {
        "base_url": os.getenv("LLM_BASE_URL", ""),
        "model": os.getenv("LLM_MODEL", ""),
        "api_key_env": "LLM_API_KEY",
    },
}

config = PROVIDER_CONFIGS[PROVIDER]

# You can also paste your key directly here instead of using .env
API_KEY = os.getenv(config["api_key_env"]) or "paste-your-key-here"
BASE_URL = config["base_url"]
MODEL = config["model"]

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL}")
print(f"Base URL : {BASE_URL}")
print(f"API key  : {'set' if API_KEY and API_KEY != 'paste-your-key-here' else 'NOT SET'}")

## 2. Choose an Image

In [1]:
from IPython.display import Image, display

# Change this to any image in your downloads/ folder
IMAGE_PATH = PROJECT_ROOT / "downloads" / "screenshots" / "ahu_02c.png"

# List available screenshots if you want to pick a different one
screenshots = sorted((PROJECT_ROOT / "downloads").rglob("*.png"))
print("Available images:")
for p in screenshots:
    print(f"  {p.relative_to(PROJECT_ROOT)}")

print(f"\nSelected: {IMAGE_PATH.relative_to(PROJECT_ROOT)}")
display(Image(filename=str(IMAGE_PATH), width=600))

NameError: name 'PROJECT_ROOT' is not defined

## 3. Write the Prompt

In [ ]:
SYSTEM_PROMPT = """
You are an HVAC equipment extraction assistant.
Inspect the attached BMS screenshot and extract every distinct, clearly visible
HVAC equipment identifier.

Only return equipment labels beginning with: AHU, VAVRH, VAV, FPTU, OAVAV, FCU.
Return JSON only in this format:
{"equipment": [{"raw_label": "...", "equipment_type": "...", "confidence": 0.0}]}
""".strip()

USER_PROMPT = "Extract all HVAC equipment identifiers visible in this image."

## 4. Send to the Vision Model

In [ ]:
import json
import httpx

def encode_image(image_path: Path) -> tuple[str, str]:
    suffix = image_path.suffix.lower()
    mime = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg"}.get(suffix.lstrip("."), "image/png")
    data = base64.b64encode(image_path.read_bytes()).decode()
    return mime, data


def call_vision_api(image_path: Path, system_prompt: str, user_prompt: str) -> dict:
    mime, data = encode_image(image_path)
    payload = {
        "model": MODEL,
        "max_tokens": 2048,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}},
                    {"type": "text", "text": user_prompt},
                ],
            },
        ],
    }
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    response = httpx.post(
        f"{BASE_URL}/chat/completions",
        json=payload,
        headers=headers,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


print(f"Sending {IMAGE_PATH.name} to {PROVIDER} ({MODEL})...")
response = call_vision_api(IMAGE_PATH, SYSTEM_PROMPT, USER_PROMPT)
print("Done.")

## 5. View the Response

In [ ]:
raw_content = response["choices"][0]["message"]["content"]
print("Raw model output:")
print(raw_content)

In [ ]:
import re

# Strip markdown fences if the model wrapped the JSON
clean = re.sub(r"^```[a-z]*\n?", "", raw_content.strip(), flags=re.MULTILINE)
clean = re.sub(r"```$", "", clean.strip())

try:
    parsed = json.loads(clean)
    print(f"\nExtracted {len(parsed.get('equipment', []))} equipment item(s):\n")
    for item in parsed.get("equipment", []):
        print(f"  {item.get('equipment_type', '?'):10}  {item.get('raw_label', '?'):25}  confidence={item.get('confidence', '?')}")
except json.JSONDecodeError as e:
    print(f"Could not parse JSON: {e}")
    print("Raw output was:")
    print(raw_content)

## 6. Usage Stats

In [ ]:
usage = response.get("usage", {})
print(f"Prompt tokens    : {usage.get('prompt_tokens', 'n/a')}")
print(f"Completion tokens: {usage.get('completion_tokens', 'n/a')}")
print(f"Total tokens     : {usage.get('total_tokens', 'n/a')}")